# 07b - Database Check and Retrieval Test

In [1]:
!pip install -q faiss-cpu sentence-transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.8 MB/s eta 0:00:00


In [2]:
import os, glob
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
import torch

## 1. Konfigurasi

In [3]:
LOCAL_CHUNKS_DIR = "/kaggle/input/notebooks/muhammadnabilul/04-chunking-and-splitting/chunks"
HF_REPO_ID  = "Makaareeem/publikasi-rag-knowledge-base"
MODEL_ID    = "nomic-ai/nomic-embed-text-v1.5"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
TOP_K       = 5
print(f"Device: {DEVICE}")

Device: cuda


## 2. Load Artefak dari HF

In [4]:
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_Faiss"))

index_path = hf_hub_download(repo_id=HF_REPO_ID, filename="faiss_index.bin", repo_type="dataset")
meta_path  = hf_hub_download(repo_id=HF_REPO_ID, filename="chunk_metadata.jsonl", repo_type="dataset")

index = faiss.read_index(index_path)
with open(meta_path) as f:
    meta = [json.loads(line) for line in f]

print(f"index.ntotal : {index.ntotal}")
print(f"metadata rows: {len(meta)}")

faiss_index.bin:   0%|          | 0.00/71.7M [00:00<?, ?B/s]

chunk_metadata.jsonl:   0%|          | 0.00/43.9M [00:00<?, ?B/s]

index.ntotal : 23336
metadata rows: 23336


## 3. Load Model

In [5]:
model = SentenceTransformer(MODEL_ID, trust_remote_code=True, device=DEVICE)

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

<All keys matched successfully>


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [6]:
## 4. Cross-check Penyimpanan (Lokal NB04 vs HF chunk_metadata)

In [7]:
hf_ids = set(c["chunk_id"] for c in meta)
print(f"HF chunk_metadata: {len(hf_ids)} unique chunk_id")

local_files = sorted(glob.glob(os.path.join(LOCAL_CHUNKS_DIR, "*.json")))
print(f"Local chunks/: {len(local_files)} file")

if local_files:
    with open(local_files[0], encoding="utf-8") as f:
        sample = json.load(f)
    print("Tipe isi file pertama:", type(sample))
    if isinstance(sample, list) and sample:
        print("Keys entri pertama:", list(sample[0].keys()))
        print("Contoh entri pertama:", sample[0])
    elif isinstance(sample, dict):
        print("Top-level keys:", list(sample.keys()))

HF chunk_metadata: 23158 unique chunk_id
Local chunks/: 47 file
Tipe isi file pertama: <class 'list'>
Keys entri pertama: ['chunk_id', 'context_hash', 'pipeline_version', 'document_title', 'document_year', 'source_file', 'bps_url', 'section_id', 'section_title', 'section_level', 'section_source', 'page_start', 'page_end', 'chunk_page_start', 'chunk_page_end', 'source_pages', 'source_chunk_ids', 'chunk_index', 'total_chunks_in_section', 'is_first_chunk', 'is_last_chunk', 'table_count', 'figure_count', 'chunk_type', 'qa_candidate', 'estimated_qa_count', 'text', 'char_count', 'word_count', 'token_estimate']
Contoh entri pertama: {'chunk_id': 'analisis-petty-corruption-hasil-spak-2020---2024__sec_002__c001', 'context_hash': '3054b63c9e2a7db49c5bea3f7c16be563fbf85dd3bd3bc57f1dcfb0e6f7af4ad', 'pipeline_version': 'chunking_v2', 'document_title': 'Analisis Petty Corruption Hasil SPAK 2020–2024', 'document_year': '2025', 'source_file': 'analisis-petty-corruption-hasil-spak-2020---2024.pdf', 'bp

In [8]:
local_ids = set()
for fp in local_files:
    with open(fp, encoding="utf-8") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else data.get("chunks", data.get("section_chunks", []))
    for item in items:
        cid = item.get("chunk_id") or item.get("id")
        if cid:
            local_ids.add(cid)

print(f"Local chunk_id terkumpul: {len(local_ids)}")
print(f"Identik dengan HF: {local_ids == hf_ids}")
only_in_hf    = hf_ids - local_ids
only_in_local = local_ids - hf_ids
print(f"Hanya di HF     : {len(only_in_hf)}", list(only_in_hf)[:5])
print(f"Hanya di lokal  : {len(only_in_local)}", list(only_in_local)[:5])

Local chunk_id terkumpul: 23158
Identik dengan HF: True
Hanya di HF     : 0 []
Hanya di lokal  : 0 []


In [9]:
def is_valid_chunk_id(cid):
    return cid in hf_ids

sample_test_id = "analisis-petty-corruption-hasil-spak-2020---2024__sec_083__c001"
print(f"Spot-check {sample_test_id} valid: {is_valid_chunk_id(sample_test_id)}")

Spot-check analisis-petty-corruption-hasil-spak-2020---2024__sec_083__c001 valid: True


## 5. Retrieval Test

In [10]:
def retrieve(query, k=TOP_K):
    q_vec = model.encode(
        [f"search_query: {query}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    scores, ids = index.search(q_vec, k)
    results = []
    for score, fid in zip(scores[0], ids[0]):
        c = meta[fid]
        results.append({
            "rank":           len(results) + 1,
            "score":          round(float(score), 4),
            "chunk_id":       c["chunk_id"],
            "document_title": c["document_title"],
            "document_year":  c["document_year"],
            "page_start":     c["page_start"],
            "bps_url":        c["bps_url"],
            "text_preview":   c["text"][:200],
        })
    return results

In [11]:
queries = [
    "Berapa tingkat pengangguran terbuka di Indonesia?",
    "Bagaimana tren kemiskinan di Indonesia 2023?",
    "Apa indikator pembangunan manusia di Indonesia?",
]

for q in queries:
    print(f"\nQuery: {q}")
    print("─" * 70)
    for r in retrieve(q):
        print(f"  [{r['rank']}] score={r['score']}  {r['chunk_id']}")
        print(f"       {r['document_title']} ({r['document_year']}) | hal {r['page_start']}")
        print(f"       {r['text_preview']}...")
        print()


Query: Berapa tingkat pengangguran terbuka di Indonesia?
──────────────────────────────────────────────────────────────────────
  [1] score=0.7857  statistik-pemuda-indonesia-2025__sec_066__c001
       Statistik Pemuda Indonesia 2025 (2025) | hal 47
       Tingkat Pengangguran Terbuka (TPT)
adalah persentase angkatan kerja yang tidak mempunyai pekerjaan, sedang aktif mencari pekerjaan atau mempersiapkan usaha, mereka yang putus asa mencari pekerjaan dan...

  [2] score=0.7855  statistik-pemuda-indonesia-2024__sec_064__c001
       Statistik Pemuda Indonesia 2024 (2024) | hal 55
       Tingkat Pengangguran Terbuka (TPT)
adalah persentase angkatan kerja yang tidak mempunyai pekerjaan, sedang aktif mencari pekerjaan atau mempersiapkan usaha, mereka yang putus asa mencari
liii

pekerja...

  [3] score=0.7833  statistik-pemuda-indonesia-2023__5__c007
       Statistik Pemuda Indonesia 2023 (2023) | hal 263
       pengangguran terdidik. Tingginya tingkat pengangguran pada pemuda terdidik ters